# ECOS Catalog Analysis
Ultrasonic characterisation and density measurements of PVA hydrogel phantoms.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ecos_loader lives next to this notebook
sys.path.insert(0, str(Path().resolve()))
import ecos_loader

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
})

base_dir = Path('../database')
df = ecos_loader.build_catalog(base_dir)

for col in ('pva_pct', 'pg_pct', 'cycle'):
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Shape: {df.shape}')
print('Conditions:')
df[['pva_pct', 'pg_pct']].drop_duplicates().sort_values(['pva_pct', 'pg_pct'])

## 2. Summary Table — Mean ± Std by Condition

In [ ]:
metrics = ['US_Cl', 'DENS_density_gcm3', 'Z', 'M_GPa']
col_labels = {
    'US_Cl':             'Cl (m/s)',
    'DENS_density_gcm3': 'density (g/cm³)',
    'Z':                 'Z (Rayl)',
    'M_GPa':             'M (GPa)',
}

summary = df.groupby(['pva_pct', 'pg_pct'])[metrics].agg(['mean', 'std', 'count']).round(4)
summary.index.names = ['PVA (%)', 'PG (%)']
summary.columns = pd.MultiIndex.from_tuples(
    [(col_labels[m], s) for m, s in summary.columns],
    names=['Metric', 'Stat'],
)
summary

## 3. Boxplot: Longitudinal Sound Speed by Condition

In [ ]:
conditions = (
    df[['pva_pct', 'pg_pct']]
    .drop_duplicates()
    .sort_values(['pva_pct', 'pg_pct'])
    .reset_index(drop=True)
)
labels = [
    f'PVA {int(r.pva_pct)}%\nPG {int(r.pg_pct)}%'
    for _, r in conditions.iterrows()
]
data_cl = [
    df[(df['pva_pct'] == r.pva_pct) & (df['pg_pct'] == r.pg_pct)]['US_Cl'].dropna().values
    for _, r in conditions.iterrows()
]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.6), 5))
bp = ax.boxplot(data_cl, labels=labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.7)
ax.set_xlabel('Condition')
ax.set_ylabel('Longitudinal velocity $C_l$ (m/s)')
ax.set_title('Longitudinal sound speed by condition')
plt.tight_layout()
plt.savefig(fig_dir / 'boxplot_Cl.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Boxplot: Density by Condition

In [ ]:
data_dens = [
    df[(df['pva_pct'] == r.pva_pct) & (df['pg_pct'] == r.pg_pct)]['DENS_density_gcm3'].dropna().values
    for _, r in conditions.iterrows()
]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.6), 5))
bp = ax.boxplot(data_dens, labels=labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('coral')
    patch.set_alpha(0.7)
ax.set_xlabel('Condition')
ax.set_ylabel('Density (g/cm³)')
ax.set_title('Density by condition')
plt.tight_layout()
plt.savefig(fig_dir / 'boxplot_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sound Speed vs PG Content

In [ ]:
grp_cl = df.groupby(['pva_pct', 'pg_pct']).agg(
    Cl_mean=('US_Cl', 'mean'),
    Cl_std=('US_Cl',  'std'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
for pva in sorted(grp_cl['pva_pct'].unique()):
    sub = grp_cl[grp_cl['pva_pct'] == pva].sort_values('pg_pct')
    ax.errorbar(
        sub['pg_pct'], sub['Cl_mean'], yerr=sub['Cl_std'],
        marker='o', capsize=5, linewidth=1.5, label=f'PVA {int(pva)}%',
    )
ax.set_xlabel('PG content (%)')
ax.set_ylabel('Mean $C_l$ (m/s)')
ax.set_title('Longitudinal sound speed vs PG content')
ax.legend(title='PVA %')
plt.tight_layout()
plt.savefig(fig_dir / 'lineplot_Cl_vs_PG.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Density vs PG Content

In [ ]:
grp_dens = df.groupby(['pva_pct', 'pg_pct']).agg(
    d_mean=('DENS_density_gcm3', 'mean'),
    d_std=('DENS_density_gcm3',  'std'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
for pva in sorted(grp_dens['pva_pct'].unique()):
    sub = grp_dens[grp_dens['pva_pct'] == pva].sort_values('pg_pct')
    ax.errorbar(
        sub['pg_pct'], sub['d_mean'], yerr=sub['d_std'],
        marker='o', capsize=5, linewidth=1.5, label=f'PVA {int(pva)}%',
    )
ax.set_xlabel('PG content (%)')
ax.set_ylabel('Mean density (g/cm³)')
ax.set_title('Density vs PG content')
ax.legend(title='PVA %')
plt.tight_layout()
plt.savefig(fig_dir / 'lineplot_density_vs_PG.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sound Speed vs Density

In [ ]:
pva_vals = sorted(df['pva_pct'].dropna().unique())
pg_vals  = sorted(df['pg_pct'].dropna().unique())

cmap    = plt.cm.tab10
colors  = {pva: cmap(i / max(len(pva_vals) - 1, 1)) for i, pva in enumerate(pva_vals)}
markers = ['o', 's', '^', 'D', 'v', 'P', '*']
mmap    = {pg: markers[i % len(markers)] for i, pg in enumerate(pg_vals)}

fig, ax = plt.subplots(figsize=(7, 5))
for pva in pva_vals:
    for pg in pg_vals:
        sub = df[(df['pva_pct'] == pva) & (df['pg_pct'] == pg)]
        if sub.empty:
            continue
        ax.scatter(
            sub['DENS_density_gcm3'], sub['US_Cl'],
            color=colors[pva], marker=mmap[pg], s=70, zorder=3,
            label=f'PVA {int(pva)}%, PG {int(pg)}%',
        )
ax.set_xlabel('Density (g/cm³)')
ax.set_ylabel('Longitudinal velocity $C_l$ (m/s)')
ax.set_title('Sound speed vs density')
ax.legend(fontsize=9, title='Condition', loc='best')
plt.tight_layout()
plt.savefig(fig_dir / 'scatter_Cl_vs_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Acoustic Impedance and Elastic Modulus by Condition

In [ ]:
grp_zm = df.groupby(['pva_pct', 'pg_pct']).agg(
    Z_mean=('Z',     'mean'), Z_std=('Z',     'std'),
    M_mean=('M_GPa', 'mean'), M_std=('M_GPa', 'std'),
).reset_index()
labels_zm = [
    f'PVA {int(r.pva_pct)}%\nPG {int(r.pg_pct)}%'
    for _, r in grp_zm.iterrows()
]
x = np.arange(len(labels_zm))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
ax1.bar(x, grp_zm['Z_mean'], yerr=grp_zm['Z_std'], color='steelblue', alpha=0.8,
        capsize=5, error_kw={'elinewidth': 1.5})
ax1.set_xticks(x)
ax1.set_xticklabels(labels_zm)
ax1.set_ylabel('Acoustic impedance Z (Rayl)')
ax1.set_title('Acoustic impedance by condition')

ax2 = axes[1]
ax2.bar(x, grp_zm['M_mean'], yerr=grp_zm['M_std'], color='coral', alpha=0.8,
        capsize=5, error_kw={'elinewidth': 1.5})
ax2.set_xticks(x)
ax2.set_xticklabels(labels_zm)
ax2.set_ylabel('Elastic modulus M (GPa)')
ax2.set_title('Longitudinal elastic modulus by condition')

plt.tight_layout()
plt.savefig(fig_dir / 'bar_Z_M.png', dpi=150, bbox_inches='tight')
plt.show()